# KG-Commit: Online Knowledge Graph Simulation

Simulates the real deployment scenario: commits arrive one-by-one in chronological order.
For each commit the loop does exactly **four steps in order**:

```
1. extract_kg_features(G, commit)   ← query G BEFORE this commit is known
2. predict(features)                ← make a prediction
3. evaluate(prediction, true_label) ← record the result
4. update_kg(G, commit)             ← add this commit's knowledge to G
```

`G` at step 1 only knows about commits **strictly before** the current one — zero label leakage.

---

### What gets added to G per commit (by tier)

| Tier | Entities | Needs |
|------|----------|-------|
| Core | `COMMIT` `TIME` `INTERVAL` `LABEL` | local CSV |
| File | `AUTHOR` `FILE` `FILE_TYPE` `DIR` `EXTERNAL_PACKAGE` | diff_text (server) |
| Within-file | `CLASS` `FUNCTION` `FUNCTION_SIGNATURE` `VARIABLE` `DATA_TYPE` | diff_text (server) |
| Finer | `ISSUE` `BRANCH` | git repo (server) |

### KG features extracted before each commit

| Feature | Source in G |
|---------|-------------|
| `kg_project_commit_count` | count of COMMIT nodes so far |
| `kg_project_bug_rate` | fraction of past commits labelled buggy |
| `kg_author_commit_count` | AUTHOR node counter |
| `kg_author_bug_rate` | AUTHOR node counter |
| `kg_file_change_count` | sum of FILE node counters for touched files |
| `kg_file_bug_rate` | mean bug rate across touched files |
| `kg_file_unique_authors` | mean unique-author count across touched files |

## 0 — Imports & Configuration

In [ ]:
import networkx as nx
import pandas as pd
import re
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# ── Local paths (work on any machine) ─────────────────────────────────
PROJECT_NAME = "apache_groovy"
LOCAL_CSV    = Path("../../data/apachejit/projects/apache_groovy.csv")

# ── SERVER-SIDE PLACEHOLDERS — set these before running on the server ─
DIFF_TEXT_CSV = Path("/path/on/server/diff/apache_groovy_diff.csv")
REPO_PATH     = Path("/path/on/server/repos/apache_groovy")

print(f"NetworkX  : {nx.__version__}")
print(f"Local CSV : {LOCAL_CSV.exists()}")

## 1 — Load Data

In [ ]:
# Sort by author_date — this is the commit stream order
df = pd.read_csv(LOCAL_CSV).sort_values("author_date").reset_index(drop=True)
print(f"Commits : {len(df):,}  |  {df.author_date.min()} → {df.author_date.max()}")

# diff_text CSV is server-side; fall back to empty dict if not present
if DIFF_TEXT_CSV.exists():
    _diff_df = pd.read_csv(DIFF_TEXT_CSV)
    diff_map = dict(zip(_diff_df.commit_id, _diff_df.diff_text))
    print(f"diff_text : {len(diff_map):,} commits loaded")
else:
    diff_map = {}
    print("diff_text : not found — file/author/within-file tiers will be skipped")

## 2 — diff_text Parser Functions

Pure functions — no side effects on `G`.
Each extracts one type of information from a raw `git show` string.

In [ ]:
import re
from pathlib import Path

# Real data = GitPython create_patch output: NO "diff --git" line, only
# the unified-diff "--- a/path" / "+++ b/path" pair, always immediately
# followed by an "@@" hunk header. All parsers below are written for that.

_RE_AUTHOR = re.compile(r'^Author:\s+(.+?)\s+<(.+?)>', re.MULTILINE)
_RE_SHOW   = re.compile(r'^(?:commit [0-9a-f]{7,}|Author:\s)', re.MULTILINE)

_RE_PY_IMP = re.compile(r'^\+\s*(?:from\s+([\w.]+)\s+import|import\s+([\w.]+))', re.MULTILINE)
_RE_JV_IMP = re.compile(r'^\+\s*import\s+(?:static\s+)?([\w.]+)\s*;', re.MULTILINE)
_RE_CLASS  = re.compile(r'^\+\s*(?:public\s+|private\s+|protected\s+|abstract\s+|final\s+|static\s+)*(?:class|interface|enum)\s+(\w+)', re.MULTILINE)

# Function signatures, matched in TWO places:
#   1. added "+" lines (a new/changed signature)
#   2. the "@@ ... @@ <heading>" section heading git appends — the
#      enclosing method that was modified (appears as CONTEXT, so a
#      "+"-only rule misses every modify-style commit, like your sample)
_RE_PY_FN  = re.compile(r'^\s*(?:async\s+)?def\s+(\w+)\s*\(([^)]*)\)(?:\s*->\s*([^:]+))?')
_RE_JV_FN  = re.compile(r'^\s*(?:(?:public|private|protected|static|final|synchronized|abstract|native|default)\s+)+([\w.<>\[\]]+)\s+(\w+)\s*\(([^)]*)\)')

_RE_ADDED  = re.compile(r'^\+(?!\+\+)(.*)$', re.MULTILINE)   # added code (not "+++")
_RE_HUNK   = re.compile(r'^@@ .*? @@\s*(.*)$', re.MULTILINE)  # hunk section heading

_RE_PY_VAR = re.compile(r'^\+\s*(?:self\.)?(\w+)\s*:\s*([\w][\w\[\], .]*?)\s*=', re.MULTILINE)
_RE_JV_VAR = re.compile(r'^\+\s*(?:final\s+)?(int|long|float|double|boolean|char|byte|short|String)\s+(\w+)\s*[=;]', re.MULTILINE)

_RE_ISSUE  = re.compile(r'\b([A-Z][A-Z0-9]+-\d+)\b')


def _strip_prefix(p):
    p = p.strip().strip('"')
    if p.startswith(("a/", "b/")):
        p = p[2:]
    return p


def parse_files(diff):
    """list of (filepath, 'add'|'remove'|'modify').

    A file-header pair "--- x" / "+++ y" is always immediately followed
    by an "@@" hunk; requiring that 3rd line avoids matching code lines
    that happen to start with '-- ' / '++ '.
    """
    if not isinstance(diff, str):
        return []
    lines = diff.split("\n")
    out = []
    for i in range(len(lines) - 2):
        l1, l2, l3 = lines[i], lines[i + 1], lines[i + 2]
        if l1.startswith("--- ") and l2.startswith("+++ ") and l3.startswith("@@"):
            a = _strip_prefix(l1[4:])
            b = _strip_prefix(l2[4:])
            if a == "/dev/null":
                out.append((b, "add"))
            elif b == "/dev/null":
                out.append((a, "remove"))
            else:
                out.append((b, "modify"))
    return out


def parse_author(diff):
    """(name, email) or (None, None). Only works on 'git show' output."""
    if not isinstance(diff, str):
        return None, None
    m = _RE_AUTHOR.search(diff)
    return (m.group(1), m.group(2)) if m else (None, None)


def parse_imports(diff):
    """set of top-level package names ADDED in this diff."""
    if not isinstance(diff, str):
        return set()
    pkgs = set()
    for m in _RE_PY_IMP.finditer(diff):
        pkg = (m.group(1) or m.group(2) or "").split(".")[0]
        if pkg:
            pkgs.add(pkg)
    for m in _RE_JV_IMP.finditer(diff):
        pkgs.add(m.group(1).split(".")[0])
    return pkgs


def parse_classes(diff):
    if not isinstance(diff, str):
        return []
    return [m.group(1) for m in _RE_CLASS.finditer(diff)]


def _match_fn(text):
    """Return (name, args, ret|None) if `text` holds a Py/Java signature."""
    m = _RE_PY_FN.search(text)
    if m:
        return m.group(1), m.group(2).strip(), (m.group(3) or "").strip() or None
    m = _RE_JV_FN.search(text)
    if m:
        return m.group(2), m.group(3).strip(), m.group(1).strip()
    return None


def parse_functions(diff):
    """list of (name, args, return_type|None) — from added lines AND
    from "@@ ... @@ <heading>" hunk headings (enclosing modified method).
    """
    if not isinstance(diff, str):
        return []
    out, seen = [], set()
    for m in _RE_ADDED.finditer(diff):
        hit = _match_fn(m.group(1))
        if hit and hit[0] not in seen:
            seen.add(hit[0]); out.append(hit)
    for m in _RE_HUNK.finditer(diff):
        hit = _match_fn(m.group(1))
        if hit and hit[0] not in seen:
            seen.add(hit[0]); out.append(hit)
    return out


def parse_variables(diff):
    """list of (var_name, type_name) for typed declarations ADDED."""
    if not isinstance(diff, str):
        return []
    out = []
    for m in _RE_PY_VAR.finditer(diff):
        out.append((m.group(1), m.group(2).strip()))
    for m in _RE_JV_VAR.finditer(diff):
        out.append((m.group(2), m.group(1)))
    return out


def parse_issues(diff):
    """set of JIRA-style issue IDs from the commit MESSAGE.

    Raw-diff data has no message -> returns empty. Only scans the header
    region of 'git show' output, so code tokens are never mistaken for
    issue keys.
    """
    if not isinstance(diff, str) or not _RE_SHOW.search(diff):
        return set()
    header = re.split(r'^(?:diff --git |--- )', diff, maxsplit=1, flags=re.MULTILINE)[0]
    return set(_RE_ISSUE.findall(header))


print("Parser functions ready (raw-diff aware).")

## 3 — `update_kg(G, row, diff_text, prev_cid)`

Called **after** prediction for each commit.
Adds all nodes and edges for that one commit to `G`.
Running counters on `AUTHOR` and `FILE` nodes make feature extraction O(1).

In [ ]:
def update_kg(G, row, diff, prev_cid=None):
    """
    Add one commit's knowledge to G.  Call AFTER prediction.

    G         – nx.MultiDiGraph, the running knowledge graph
    row       – pd.Series, one row from the time-sorted CSV
    diff      – str | None, raw git-show output for this commit
    prev_cid  – node-id string of the previous commit in this project, or None
    """
    cid = f"commit:{row.commit_id}"
    ts  = int(row.author_date)

    # ── TIER 1: Core ──────────────────────────────────────────────────

    G.add_node(cid, type="COMMIT", commit_id=row.commit_id,
               project=row.project, year=int(row.year), author_date=ts)

    tid = f"time:{ts}"
    if not G.has_node(tid):
        G.add_node(tid, type="TIME", datetime=ts)
    G.add_edge(cid, tid, rel="at_time")

    # INTERVAL between previous commit and this one
    if prev_cid is not None:
        prev_ts = G.nodes[prev_cid]["author_date"]
        iid = f"interval:{prev_cid}_{row.commit_id}"
        G.add_node(iid, type="INTERVAL", project=row.project)
        G.add_edge(iid, f"time:{prev_ts}", rel="begin")
        G.add_edge(iid, tid,              rel="end")
        G.add_edge(prev_cid, iid,          rel="duration")

    status = -1 if row.buggy else (1 if row.fix else 0)
    lid = f"label:{status}"
    if not G.has_node(lid):
        G.add_node(lid, type="LABEL", status=status)
    G.add_edge(cid, lid, rel="is")

    # ── TIER 2: Author ────────────────────────────────────────────────

    _, email = parse_author(diff)
    aid = f"author:{email or 'unknown'}"
    if not G.has_node(aid):
        G.add_node(aid, type="AUTHOR", email=email, commit_count=0, bug_count=0)
    G.add_edge(cid, aid, rel="by")
    G.nodes[aid]["commit_count"] += 1
    if row.buggy:
        G.nodes[aid]["bug_count"] += 1

    # Author interval: extend end pointer each time they commit
    auth_iid = f"interval:author:{email or 'unknown'}"
    if G.has_node(auth_iid):
        for _, t, d in list(G.out_edges(auth_iid, data=True)):
            if d.get("rel") == "end":
                G.remove_edge(auth_iid, t)
        G.add_edge(auth_iid, tid, rel="end")
    else:
        G.add_node(auth_iid, type="INTERVAL")
        G.add_edge(auth_iid, tid, rel="begin")
        G.add_edge(auth_iid, tid, rel="end")
        G.add_edge(aid, auth_iid, rel="duration")

    # ── TIER 2: Files, dirs, file types ──────────────────────────────

    files = parse_files(diff)
    for filepath, change_type in files:
        p    = Path(filepath)
        fid  = f"file:{filepath}"
        ftid = f"filetype:{p.suffix or 'none'}"

        if not G.has_node(fid):
            G.add_node(fid, type="FILE", name=p.name, path=filepath,
                       change_count=0, bug_count=0, authors=set())
        G.add_edge(cid, fid, rel=change_type)
        G.nodes[fid]["change_count"] += 1
        if row.buggy:
            G.nodes[fid]["bug_count"] += 1
        G.nodes[fid]["authors"].add(email or "unknown")

        if not G.has_node(ftid):
            G.add_node(ftid, type="FILE_TYPE", format=p.suffix)
        G.add_edge(fid, ftid, rel="type")

        parents = list(reversed(list(p.parents)))
        for i, part in enumerate(parents):
            if str(part) in (".", ""):
                continue
            did = f"dir:{part}"
            if not G.has_node(did):
                G.add_node(did, type="DIR", name=part.name, path=str(part))
            if i == len(parents) - 1:
                G.add_edge(fid, did, rel="parent")
            if i > 0:
                pdid = f"dir:{parents[i-1]}"
                if G.has_node(pdid):
                    G.add_edge(did, pdid, rel="parent")
                    G.add_edge(pdid, did, rel="child")

        # File interval: extend end pointer each time it is touched
        file_iid = f"interval:file:{filepath}"
        if G.has_node(file_iid):
            for _, t, d in list(G.out_edges(file_iid, data=True)):
                if d.get("rel") == "end":
                    G.remove_edge(file_iid, t)
            G.add_edge(file_iid, tid, rel="end")
        else:
            G.add_node(file_iid, type="INTERVAL")
            G.add_edge(file_iid, tid, rel="begin")
            G.add_edge(file_iid, tid, rel="end")
            G.add_edge(fid, file_iid, rel="duration")

    # ── TIER 3: External packages ─────────────────────────────────────

    for pkg in parse_imports(diff):
        pid = f"extpkg:{pkg}"
        if not G.has_node(pid):
            G.add_node(pid, type="EXTERNAL_PACKAGE", name=pkg)
        for fp, _ in files:
            G.add_edge(f"file:{fp}", pid, rel="imports")

    # ── TIER 4: Within-file entities ──────────────────────────────────

    fids = [f"file:{fp}" for fp, _ in files]

    for cls in parse_classes(diff):
        clid = f"class:{cls}"
        if not G.has_node(clid): G.add_node(clid, type="CLASS", name=cls)
        for fid in fids: G.add_edge(fid, clid, rel="contains")

    for fn, args, ret in parse_functions(diff):
        fnid   = f"func:{fn}"
        sig_id = f"sig:{fn}({args})->{ret}"
        if not G.has_node(fnid): G.add_node(fnid, type="FUNCTION", name=fn)
        G.add_node(sig_id, type="FUNCTION_SIGNATURE", args=args, returns=ret)
        G.add_edge(fnid, sig_id, rel="has_signature")
        for fid in fids: G.add_edge(fid, fnid, rel="contains")

    for var, dtype in parse_variables(diff):
        vid  = f"var:{var}"
        dtid = f"dtype:{dtype}"
        if not G.has_node(vid):  G.add_node(vid,  type="VARIABLE",  name=var)
        if not G.has_node(dtid): G.add_node(dtid, type="DATA_TYPE", id=dtype)
        G.add_edge(vid, dtid, rel="has_type")
        for fid in fids: G.add_edge(fid, vid, rel="contains")

    # ── TIER 5: Issues (from commit message header) ───────────────────

    for issue_id in parse_issues(diff):
        inode = f"issue:{issue_id}"
        if not G.has_node(inode): G.add_node(inode, type="ISSUE", id=issue_id)
        G.add_edge(cid, inode, rel="for")

    # ── TIER 5: Branch (placeholder — needs git repo) ─────────────────
    # import subprocess
    # branches = subprocess.check_output(
    #     ["git", "-C", str(REPO_PATH), "branch", "--contains", row.commit_id],
    #     text=True).strip().splitlines()
    # for b in branches:
    #     b = b.strip().lstrip("* ")
    #     if not b: continue
    #     bid = f"branch:{b}"
    #     if not G.has_node(bid): G.add_node(bid, type="BRANCH", id=b)
    #     G.add_edge(cid, bid, rel="in")


print("update_kg() ready.")

### 2.1 — Parser Sanity Test

Applies every parser to a **real** `apache_groovy` diff sample so you can inspect exactly what each one extracts before running the full pipeline.

In [ ]:
# Real sample from the apache_groovy diff_text column (GitPython create_patch).
SAMPLE_DIFF = r"""--- a/src/main/org/codehaus/groovy/runtime/DefaultGroovyMethods.java
+++ b/src/main/org/codehaus/groovy/runtime/DefaultGroovyMethods.java
@@ -65,7 +65,6 @@ STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE)
 import java.sql.ResultSet;
 import java.sql.SQLException;
 import java.util.ArrayList;
-import java.util.Arrays;
 import java.util.Collection;
@@ -655,23 +654,23 @@ public static List reverse(List self) {
     public static List plus(List left, Collection right) {
-            List answer = new ArrayList(left.size()+right.size());
+        List answer = new ArrayList(left.size() + right.size());
@@ -679,14 +678,15 @@ public static List multiply(List self, Number factor) {
     public static List intersect(List left, Collection right) {
-        if (left.size()==0)
+        if (left.size() == 0)
@@ -698,7 +698,7 @@ public static List intersect(List left, Collection right) {
     public static List minus(List self, Collection removeMe) {
-        if (self.size() ==0 )
+        if (self.size() == 0 )
@@ -728,13 +729,15 @@ public static List minus(List self, Collection removeMe) {
-           List answer=new LinkedList();
+           List answer = new LinkedList();
@@ -751,7 +754,7 @@ public static List flatten(List self) {
-            Object element=iter.next();
+            Object element = iter.next();
@@ -767,23 +770,23 @@ else if (element instanceof Map) {
     private static boolean sameType(Collection[] cols)
@@ -867,11 +870,14 @@ else if (isLong(left) || isLong(right)) {
     public static Number power(Number self, Number exponent) {
-        double answer=Math.pow(self.doubleValue(), exponent.doubleValue());
+        double answer = Math.pow(self.doubleValue(),
+                exponent.doubleValue());
-        int size=factor.intValue();
+        int size = factor.intValue();
"""

print("=" * 64)
print("PARSER SANITY TEST  (on a real apache_groovy diff sample)")
print("=" * 64)

print("\nparse_files:")
for fp, ch in parse_files(SAMPLE_DIFF):
    print(f"   [{ch:6}] {fp}")

print("\nparse_author:")
print("  ", parse_author(SAMPLE_DIFF), " <- (None, None) expected: raw diff has no header")

print("\nparse_imports (added '+import' only):")
print("  ", parse_imports(SAMPLE_DIFF) or "set()  <- sample only removes an import")

print("\nparse_classes:")
print("  ", parse_classes(SAMPLE_DIFF) or "[]  <- no class added in sample")

print("\nparse_functions (added lines + '@@ ...@@ heading' enclosing methods):")
for nm, args, ret in parse_functions(SAMPLE_DIFF):
    print(f"   {nm:12} args=({args})  ret={ret}")

print("\nparse_variables (typed declarations on added lines):")
for v, t in parse_variables(SAMPLE_DIFF):
    print(f"   {t:8} {v}")

print("\nparse_issues:")
print("  ", parse_issues(SAMPLE_DIFF) or "set()  <- no commit message in raw diff")

# ── What update_kg would add for this single commit ───────────────────
print("\n" + "=" * 64)
print("update_kg() DRY RUN on the sample")
print("=" * 64)
_g = nx.MultiDiGraph()
_row = pd.Series({
    "commit_id": "deadbeefcafe", "project": "apache/groovy",
    "buggy": False, "fix": True, "year": 2005, "author_date": 1111111111,
})
update_kg(_g, _row, SAMPLE_DIFF, prev_cid=None)
print(pd.Series(nx.get_node_attributes(_g, "type")).value_counts().to_string())
print(f"\n  total: {_g.number_of_nodes()} nodes, {_g.number_of_edges()} edges")

## 4 — `extract_kg_features(G, row, diff)`

Called **before** `update_kg` — `G` contains only past commits.
Returns a flat dict of floats ready to concatenate with the handcrafted feature vector.

In [ ]:
def extract_kg_features(G, row, diff):
    """
    Query G for features about this commit BEFORE it is added.
    All values default to 0.0 if the relevant history does not exist yet.
    """
    feats = {}

    # ── Project-level ─────────────────────────────────────────────────
    commit_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "COMMIT"]
    n = len(commit_nodes)
    feats["kg_project_commit_count"] = float(n)
    if n > 0:
        n_bug = sum(
            1 for cn in commit_nodes
            if any(d.get("rel") == "is" and v == "label:-1"
                   for _, v, d in G.out_edges(cn, data=True))
        )
        feats["kg_project_bug_rate"] = n_bug / n
    else:
        feats["kg_project_bug_rate"] = 0.0

    # ── Author-level ──────────────────────────────────────────────────
    _, email = parse_author(diff)
    aid = f"author:{email or 'unknown'}"
    ad  = G.nodes[aid] if G.has_node(aid) else {}
    ac  = ad.get("commit_count", 0)
    bc  = ad.get("bug_count",    0)
    feats["kg_author_commit_count"] = float(ac)
    feats["kg_author_bug_rate"]     = bc / ac if ac > 0 else 0.0

    # ── File-level ────────────────────────────────────────────────────
    change_counts, bug_rates, uniq_authors = [], [], []
    for fp, _ in parse_files(diff):
        fid = f"file:{fp}"
        if not G.has_node(fid):
            continue
        fd  = G.nodes[fid]
        cc  = fd.get("change_count", 0)
        bcc = fd.get("bug_count",    0)
        change_counts.append(cc)
        bug_rates.append(bcc / cc if cc > 0 else 0.0)
        uniq_authors.append(len(fd.get("authors", set())))

    feats["kg_file_change_count"]   = float(sum(change_counts))
    feats["kg_file_bug_rate"]       = sum(bug_rates)   / len(bug_rates)   if bug_rates   else 0.0
    feats["kg_file_unique_authors"] = sum(uniq_authors) / len(uniq_authors) if uniq_authors else 0.0

    return feats


print("extract_kg_features() ready.")

## 5 — Online Simulation Loop

Initialise an empty graph and a fresh online classifier.
The four-step loop runs over commits in strict chronological order.

In [ ]:
# Handcrafted features available from the CSV
HC_COLS = ["la", "ld", "nf", "nd", "ns", "ent", "ndev", "age", "nuc", "aexp", "arexp", "asexp"]

# KG features in fixed order
KG_COLS = [
    "kg_project_commit_count",
    "kg_project_bug_rate",
    "kg_author_commit_count",
    "kg_author_bug_rate",
    "kg_file_change_count",
    "kg_file_bug_rate",
    "kg_file_unique_authors",
]

# ── Initialise empty graph and model ─────────────────────────────────
G            = nx.MultiDiGraph()
model        = SGDClassifier(loss="log_loss", max_iter=1, warm_start=True, random_state=42)
scaler       = StandardScaler()
results      = []          # accumulates {commit_id, true_label, predicted}
prev_cid_map = {}          # project → node-id of last commit seen
WARMUP       = 50          # commits before we start predicting

print(f"Starting online simulation over {len(df):,} commits …")

In [ ]:
for i, row in df.iterrows():

    diff       = diff_map.get(row.commit_id)     # None if server CSV not loaded
    true_label = int(row.buggy)                  # 1 = buggy, 0 = clean

    # ── STEP 1: Extract KG features ── G knows nothing about this commit yet ──
    kg   = extract_kg_features(G, row, diff)
    hc   = [float(row[c]) for c in HC_COLS]
    x    = np.array(hc + [kg[k] for k in KG_COLS], dtype=float)

    # ── STEP 2: Predict ───────────────────────────────────────────────────────
    predicted = None
    if i >= WARMUP:
        x_sc      = scaler.transform(x.reshape(1, -1))
        predicted = int(model.predict(x_sc)[0])

    # ── STEP 3: Record result ─────────────────────────────────────────────────
    results.append({
        "commit_id":   row.commit_id,
        "author_date": row.author_date,
        "true_label":  true_label,
        "predicted":   predicted,
    })

    # ── STEP 4: Update KG ─────────────────────────────────────────────────────
    prev_cid = prev_cid_map.get(row.project)
    update_kg(G, row, diff, prev_cid=prev_cid)
    prev_cid_map[row.project] = f"commit:{row.commit_id}"

    # ── Online model update ───────────────────────────────────────────────────
    scaler.partial_fit(x.reshape(1, -1))
    x_sc = scaler.transform(x.reshape(1, -1))
    model.partial_fit(x_sc, [true_label], classes=[0, 1])

    if (i + 1) % 1000 == 0:
        done = i + 1
        print(f"  {done:,} / {len(df):,} commits processed …")

print(f"\nDone. G has {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.")

## 6 — Evaluation

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["predicted"])
y_true = results_df.true_label.astype(int)
y_pred = results_df.predicted.astype(int)

print(f"Evaluated on {len(results_df):,} commits (after {WARMUP}-commit warm-up)\n")
print(classification_report(y_true, y_pred, target_names=["clean", "buggy"]))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred))

## 7 — Graph Inspection & Export

In [ ]:
# Node / edge type breakdown
node_counts = pd.Series(nx.get_node_attributes(G, "type")).value_counts()
edge_counts = pd.Series([d.get("rel") for _, _, d in G.edges(data=True)]).value_counts()

print(f"Nodes : {G.number_of_nodes():,}   Edges : {G.number_of_edges():,}\n")
print("── Node types ──────────────")
print(node_counts.to_string())
print("\n── Edge relation types ─────")
print(edge_counts.head(20).to_string())

In [ ]:
# Sample query: all outgoing edges from the last commit
last_cid = f"commit:{df.commit_id.iloc[-1]}"
print(f"Neighbourhood of {last_cid}:\n")
for _, nb, d in G.out_edges(last_cid, data=True):
    print(f"  --[{d['rel']}]--> {nb}  ({G.nodes[nb].get('type', '?')})")

In [ ]:
OUTPUT_DIR = Path("../../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GraphML is not compatible with Python sets — convert authors to string first
for _, d in G.nodes(data=True):
    if "authors" in d:
        d["authors"] = ",".join(sorted(d["authors"]))

graphml_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg.graphml"
nx.write_graphml(G, graphml_path)
print(f"GraphML : {graphml_path}")

json_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg.json"
with open(json_path, "w") as f:
    json.dump(nx.node_link_data(G), f, indent=2)
print(f"JSON    : {json_path}")

results_path = OUTPUT_DIR / f"{PROJECT_NAME}_online_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Results : {results_path}")

---
## 8 — Step-by-Step KG Growth Inspector

Builds a **fresh** `G_insp` (independent of the main simulation) and visualizes after:

| Step | Commits in G |
|:----:|:------------:|
| 0–6  | one at a time (7 plots) |
| 26   | +20 |
| 60   | +34 |

Each checkpoint shows a **2D** (matplotlib) and **3D** (plotly, interactive) view.
Node colors are consistent across all plots.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go

NODE_COLORS = {
    'COMMIT':             '#4C72B0',
    'TIME':               '#55A868',
    'INTERVAL':           '#FF8C00',
    'LABEL':              '#C44E52',
    'AUTHOR':             '#8172B2',
    'FILE':               '#937860',
    'FILE_TYPE':          '#DA8BC3',
    'DIR':                '#8C8C8C',
    'EXTERNAL_PACKAGE':   '#CCB974',
    'CLASS':              '#64B5CD',
    'FUNCTION':           '#1F77B4',
    'FUNCTION_SIGNATURE': '#AEC7E8',
    'VARIABLE':           '#98DF8A',
    'DATA_TYPE':          '#FF9896',
    'ISSUE':              '#FFBB78',
    'BRANCH':             '#17BECF',
}

def _groups(G):
    g = defaultdict(list)
    for n, d in G.nodes(data=True):
        g[d.get('type', 'UNKNOWN')].append(n)
    return g

def _short(nid, mx=18):
    s = nid.split(':', 1)[-1]
    return s[:mx] + '...' if len(s) > mx else s

def _stats(G):
    g = _groups(G)
    print(f'  nodes={G.number_of_nodes()}  edges={G.number_of_edges()}')
    print('  ' + '  '.join(f'{t}={len(ns)}' for t, ns in sorted(g.items())))


def draw_kg_2d(G, title=''):
    if G.number_of_nodes() == 0:
        print(f'[2D] {title}: empty graph.'); return
    n   = G.number_of_nodes()
    pos = nx.spring_layout(G, seed=42, k=3.0/(n**0.5) if n>1 else 2.0, iterations=60)
    grp = _groups(G)
    fig, ax = plt.subplots(figsize=(14, 8))
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.2, arrows=True,
                           edge_color='#999', width=0.6,
                           connectionstyle='arc3,rad=0.08')
    sz = max(20, 400 - n*4)
    for t, ns in grp.items():
        nx.draw_networkx_nodes(G, pos, nodelist=ns, ax=ax,
                               node_color=NODE_COLORS.get(t, '#DDD'),
                               node_size=sz, alpha=0.9)
    if n <= 40:
        nx.draw_networkx_labels(G, pos,
                                labels={nd: _short(nd) for nd in G.nodes()},
                                ax=ax, font_size=6)
    patches = [mpatches.Patch(color=NODE_COLORS.get(t, '#DDD'),
                              label=f'{t} ({len(ns)})')
               for t, ns in sorted(grp.items())]
    ax.legend(handles=patches, loc='upper left', fontsize=7, framealpha=0.85, ncol=2)
    ax.set_title(f'{title}  |  nodes={n}  edges={G.number_of_edges()}', fontsize=11)
    ax.axis('off'); plt.tight_layout(); plt.show()


def draw_kg_3d(G, title=''):
    if G.number_of_nodes() == 0:
        print(f'[3D] {title}: empty graph.'); return
    n   = G.number_of_nodes()
    pos = nx.spring_layout(G, dim=3, seed=42, k=2.0/(n**0.5), iterations=60)
    ex, ey, ez = [], [], []
    for u, v in G.edges():
        x0,y0,z0=pos[u]; x1,y1,z1=pos[v]
        ex+=[x0,x1,None]; ey+=[y0,y1,None]; ez+=[z0,z1,None]
    traces = [go.Scatter3d(x=ex,y=ey,z=ez,mode='lines',
                           line=dict(color='#AAA',width=1),
                           hoverinfo='none',showlegend=False)]
    sz = max(3, 10 - n//20)
    for t, ns in sorted(_groups(G).items()):
        traces.append(go.Scatter3d(
            x=[pos[nd][0] for nd in ns],
            y=[pos[nd][1] for nd in ns],
            z=[pos[nd][2] for nd in ns],
            mode='markers',
            marker=dict(size=sz,color=NODE_COLORS.get(t,'#DDD'),
                        opacity=0.88,line=dict(width=0.5,color='#333')),
            text=[_short(nd,30) for nd in ns],
            hoverinfo='text+name',
            name=f'{t} ({len(ns)})',
        ))
    ax_s = dict(showgrid=False,zeroline=False,showticklabels=False,showbackground=False)
    go.Figure(
        data=traces,
        layout=go.Layout(
            title=f'{title}  |  nodes={n}  edges={G.number_of_edges()}',
            scene=dict(xaxis=ax_s,yaxis=ax_s,zaxis=ax_s),
            margin=dict(l=0,r=0,b=0,t=45),height=550,
            legend=dict(font=dict(size=9),itemsizing='constant'),
        )
    ).show()


print('draw_kg_2d / draw_kg_3d ready.')

### 8.1 — Commits 0 → 6, one at a time

In [ ]:
G_insp    = nx.MultiDiGraph()
prev_insp = None

print('=' * 55)
print('CHECKPOINT: 0 commits  (empty graph)')
print('=' * 55)

for step in range(1, 7):
    row  = df.iloc[step - 1]
    diff = diff_map.get(row.commit_id)
    update_kg(G_insp, row, diff, prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'

    lbl = 'buggy' if row.buggy else ('fix' if row.fix else 'clean')
    print()
    print('=' * 55)
    print(f'CHECKPOINT: {step} commit{"s" if step>1 else ""}'
          f'  [{row.commit_id[:10]}...  {lbl}  {row.year}]')
    print('=' * 55)
    _stats(G_insp)
    draw_kg_2d(G_insp, title=f'{step} commit{"s" if step>1 else ""}')
    draw_kg_3d(G_insp, title=f'{step} commit{"s" if step>1 else ""}')

### 8.2 — Checkpoint: 26 commits  (+20)

In [ ]:
for step in range(7, 27):
    row = df.iloc[step - 1]
    update_kg(G_insp, row, diff_map.get(row.commit_id), prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'

print('=' * 55)
print('CHECKPOINT: 26 commits  (+20 from previous)')
print('=' * 55)
_stats(G_insp)
draw_kg_2d(G_insp, title='26 commits')
draw_kg_3d(G_insp, title='26 commits')

### 8.3 — Checkpoint: 60 commits  (+34)

In [ ]:
for step in range(27, 61):
    row = df.iloc[step - 1]
    update_kg(G_insp, row, diff_map.get(row.commit_id), prev_cid=prev_insp)
    prev_insp = f'commit:{row.commit_id}'

print('=' * 55)
print('CHECKPOINT: 60 commits  (+34 from previous)')
print('=' * 55)
_stats(G_insp)
draw_kg_2d(G_insp, title='60 commits')
draw_kg_3d(G_insp, title='60 commits')